In [1]:
import sys

import pm4py

import pandas as pd
import numpy as np

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts
from model.next_event_model import ProcessLSTM, train_ProcessLSTM, validate_ProcessLSTM

### --- Preprocess dataset ---

In [2]:
set_seed(seed=42)

In [3]:
log = pm4py.read_xes("../../data/Road_Traffic_Fine_Management_Process.xes")

C:\Users\dcoralage\Downloads\counterfactual_prediction_experiments\counterfactual_env\lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
C:\Users\dcoralage\Downloads\counterfactual_prediction_experiments\counterfactual_env\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/150370 [00:00<?, ?it/s]

In [4]:
df = pm4py.convert_to_dataframe(log)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 561470 entries, 0 to 561469
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   amount                230230 non-null  float64            
 1   org:resource          150925 non-null  object             
 2   dismissal             155066 non-null  object             
 3   concept:name          561470 non-null  object             
 4   vehicleClass          150370 non-null  object             
 5   totalPaymentAmount    227971 non-null  float64            
 6   lifecycle:transition  561470 non-null  object             
 7   time:timestamp        561470 non-null  datetime64[ns, UTC]
 8   article               150370 non-null  float64            
 9   points                150370 non-null  float64            
 10  case:concept:name     561470 non-null  object             
 11  expense               103987 non-null  float64      

In [6]:
df.isnull().any()

amount                   True
org:resource             True
dismissal                True
concept:name            False
vehicleClass             True
totalPaymentAmount       True
lifecycle:transition    False
time:timestamp          False
article                  True
points                   True
case:concept:name       False
expense                  True
notificationType         True
lastSent                 True
paymentAmount            True
matricola                True
dtype: bool

In [7]:
df = df.drop(columns=['matricola'])

In [8]:
df['case:concept:name'] = df['case:concept:name'].astype('string')
df['concept:name'] = df['concept:name'].astype('string')
df['lifecycle:transition'] = df['lifecycle:transition'].astype('string')
df['org:resource'] = df['org:resource'].astype('string')
df['dismissal'] = df['dismissal'].astype('string')
df['vehicleClass'] = df['vehicleClass'].astype('string')
df['notificationType'] = df['notificationType'].astype('string')
df['lastSent'] = df['lastSent'].astype('string')

df['amount'] = df['amount'].astype(np.float32)
df['totalPaymentAmount'] = df['totalPaymentAmount'].astype(np.float32)
df['article'] = df['article'].astype(np.float32)
df['points'] = df['points'].astype(np.float32)
df['expense'] = df['expense'].astype(np.float32)
df['paymentAmount'] = df['paymentAmount'].astype(np.float32)

df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], errors='coerce')

In [9]:
df = df.sort_values(by=['case:concept:name', 'time:timestamp'], ascending=[True, True])

In [10]:
df['time_delta'] = df.groupby('case:concept:name')['time:timestamp'].diff()
df['time_delta'] = df['time_delta'].dt.total_seconds().astype(np.float32)
df['time_delta'] = df['time_delta'].fillna(0)

In [11]:
exclude_cols = ["case:concept:name", "time:timestamp"]

sorted_cols = sorted(
    [c for c in df.columns if c not in exclude_cols]
)

df = df[exclude_cols + sorted_cols]

In [12]:
df.head(20)

,case:concept:name,time:timestamp,amount,article,concept:name,dismissal,expense,lastSent,lifecycle:transition,notificationType,org:resource,paymentAmount,points,time_delta,totalPaymentAmount,vehicleClass
0,A1,2006-07-24 00:00:00+00:00,35.0,157.0,Create Fine,NIL,NaN,<NA>,complete,<NA>,561,NaN,0.0,0.0,0.0,A
1,A1,2006-12-05 00:00:00+00:00,NaN,NaN,Send Fine,<NA>,11.0,<NA>,complete,<NA>,<NA>,NaN,NaN,11577600.0,NaN,<NA>
2,A100,2006-08-02 00:00:00+00:00,35.0,157.0,Create Fine,NIL,NaN,<NA>,complete,<NA>,561,NaN,0.0,0.0,0.0,A
3,A100,2006-12-12 00:00:00+00:00,NaN,NaN,Send Fine,<NA>,11.0,<NA>,complete,<NA>,<NA>,NaN,NaN,11404800.0,NaN,<NA>
4,A100,2007-01-15 00:00:00+00:00,NaN,NaN,Insert Fine Notification,<NA>,NaN,P,complete,P,<NA>,NaN,NaN,2937600.0,NaN,<NA>
5,A100,2007-03-16 00:00:00+00:00,71.5,NaN,Add penalty,<NA>,NaN,<NA>,complete,<NA>,<NA>,NaN,NaN,5184000.0,NaN,<NA>
6,A100,2009-03-30 00:00:00+00:00,NaN,NaN,Send for Credit Collection,<NA>,NaN,<NA>,complete,<NA>,<NA>,NaN,NaN,64368000.0,NaN,<NA>
7,A10000,2007-03-09 00:00:00+00:00,36.0,157.0,Create Fine,NIL,NaN,<NA>,complete,<NA>,561,NaN,0.0,0.0,0.0,A
8,A10000,2007-07-17 00:00:00+00:00,NaN,NaN,Send Fine,<NA>,13.0,<NA>,complete,<NA>,<NA>,NaN,NaN,11232000.0,NaN,<NA>
9,A10000,2007-08-02 00:00:00+00:00,NaN,NaN,Insert Fine Notification,<NA>,NaN,P,complete,P,<NA>,NaN,NaN,1382400.0,NaN,<NA>


In [13]:
num_cases = df['case:concept:name'].nunique()
print(f"Total number of unique cases: {num_cases}")

Total number of unique cases: 150370


### --- Feature Configurations ---

In [14]:
# --- Define feature specs ---
feature_specs = {

    "time_delta": {
        "type":           "continuous",
        "level":          "event",
        "vary":           True,
        "quantile_low":   0.20,
        "quantile_high":  0.80, 
    },

    "amount": {
        "type":           "continuous",
        "level":          "event",
        "vary":           True,
        "quantile_low":   0.05,
        "quantile_high":  0.90, 
                 
    },

    "totalPaymentAmount": {
        "type":           "continuous",
        "level":          "event",
        "vary":           True,
        "quantile_low":   0.05,
        "quantile_high":  0.90, 
                 
    },

    "article": {
        "type":           "continuous",
        "level":          "event",
        "vary":           True,
                 
    },

    "points": {
        "type":           "continuous",
        "level":          "event",
        "vary":           True,
        "quantile_low":   0.05,
        "quantile_high":  0.90, 
                 
    },

    "expense": {
        "type":           "continuous",
        "level":          "event",
        "vary":           True,
                 
    },

    "paymentAmount": {
        "type":           "continuous",
        "level":          "event",
        "vary":           True,
        "quantile_low":   0.05,
        "quantile_high":  0.90, 
                 
    },

    "org:resource": {
        "type":           "categorical",
        "level":          "event",
        "vary":           True,
    },

    "dismissal": {
        "type":           "categorical",
        "level":          "event",
        "vary":           True,
    },

    "vehicleClass": {
        "type":           "categorical",
        "level":          "event",
        "vary":           True,
    },

    "notificationType": {
        "type":           "categorical",
        "level":          "event",
        "vary":           True,
    },

    "lastSent": {
        "type":           "categorical",
        "level":          "event",
        "vary":           True,
    },

    # immutable
    "concept:name": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           False
    },

    "lifecycle:transition": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           False
    },
}

In [15]:
feature_config = FeatureConfig.from_dataframe(
    df=df,
    feature_specs=feature_specs,
    activity_feature="concept:name",
    is_robust=True,
    default_quantile_low=0.05,
    default_quantile_high=0.95
)

feature_config.save()

In [16]:
# feature_config = FeatureConfig.load()

In [17]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['amount', 'article', 'concept:name', 'dismissal', 'expense', 'lastSent', 'lifecycle:transition', 'notificationType', 'org:resource', 'paymentAmount', 'points', 'time_delta', 'totalPaymentAmount', 'vehicleClass']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [0.00, 8726400.00]                       2160000.0000 quantile_derived    
amount                         continuous     event    yes    [24.00, 80.00]                           6.7000     quantile_derived    
totalPaymentAmount             continuous     event

### --- Next event prediction model ---

In [18]:
# Transform nan cols to NA
cat_cols = df.select_dtypes(include=["string"]).columns
for col in cat_cols:
    df[col] = df[col].fillna("NA").astype('string')

In [19]:
case_ids = df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = df[df["case:concept:name"].isin(train_cases)].copy()
val_df   = df[df["case:concept:name"].isin(val_cases)].copy()

In [20]:
preprocessor_artifacts = PreprocessorArtifacts.build(
    df=train_df,
    feature_config=feature_config,      
    scaler_type="robust",
)

preprocessor_artifacts.save()

In [21]:
# preprocessor_artifacts = PreprocessorArtifacts.load()

In [22]:
preprocessor_artifacts.summary()

===================PreprocessorArtifacts====================
  scaler:              RobustScaler
  encoders:            ['org:resource', 'dismissal', 'vehicleClass', 'notificationType', 'lastSent', 'concept:name', 'lifecycle:transition']
  activity_prototypes: 11 activities


In [23]:
# Transform nan cols to 0
float_cols = df.select_dtypes(include=["float32", "float64"]).columns
df[float_cols] = df[float_cols].fillna(0)
train_df[float_cols] = train_df[float_cols].fillna(0)
val_df[float_cols] = val_df[float_cols].fillna(0)

In [24]:
train_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    df=train_df,
    case_id_field="case:concept:name", 
    sort_field="time:timestamp"
)

val_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    df=val_df,
    case_id_field="case:concept:name", 
    sort_field="time:timestamp"
)

In [25]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [26]:
print(preprocessor_artifacts.get_categorical_feature_cardinality())

{'dynamic_categorical_info': {'concept:name': 11, 'dismissal': 27, 'lastSent': 4, 'lifecycle:transition': 1, 'notificationType': 3, 'org:resource': 149, 'vehicleClass': 5}, 'static_categorical_info': {}}


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [28]:
device

device(type='cuda')

In [29]:
criterion = torch.nn.CrossEntropyLoss()

In [30]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/road_fine-model_output.txt")

Epoch 020/100 | Train Loss: 0.4067 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.3925 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.3778 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.3633 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.3558 | LR: 1.00e-06
Time taken for next event model (training): 12740.111408 seconds
Time taken for next event model (validation): 7.954185 seconds
Val loss: {'loss': 0.5449521098419865, 'accuracy': 0.8076652089407191, 'f1_macro': 0.6978412623671005, 'f1_weighted': 0.7631480429300703}


In [31]:
embedding_metadata = preprocessor_artifacts.get_embedding_metadata()

model = ProcessLSTM(
    dynamic_categorical_info=embedding_metadata["dynamic_categorical_info"],
    static_categorical_info=embedding_metadata["static_categorical_info"],
    n_dynamic_continuous=embedding_metadata["n_dynamic_continuous"],
    n_static_continuous=embedding_metadata["n_static_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ProcessLSTM(
    model=model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

model.save()

In [32]:
# model = ProcessLSTM.load()

In [33]:
val_loss = validate_ProcessLSTM(
    model=model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [34]:
# --- Save processed df ---
if df["time:timestamp"].dt.tz is not None:
    df["time:timestamp"] = df["time:timestamp"].dt.tz_convert(None)
df.to_excel("../../data/road_fine.xlsx", index=False, engine="openpyxl")

In [35]:
sys.stdout = original_stdout
log_file.close()